# VoteAnomaly · análisis de la participación

Este notebook estudia los 500 registros simulados de Diana incluidos en el repositorio. Ejecuta el pipeline de Renzo y muestra los archivos y métricas que resultan del preprocesamiento, del resumen descriptivo y del detector IQR.

Instala `requirements.txt`, selecciona el kernel Python de ese entorno y ejecuta todas las celdas en orden. Puedes abrirlo desde la raíz del repositorio o desde `notebooks/`. Las salidas se guardan bajo `outputs/temp/luna/notebook/`.

## Datos y preprocesamiento del equipo

El pipeline usa `preprocess_dataset` de Diana para normalizar los identificadores y los conteos. Después calcula el cociente entre votos reportados y censo cuando el censo es positivo. Las tasas superiores a 1 se conservan. El detector y el resumen reciben solo los registros preparados. Renzo usa el CSV separado de etiquetas para calcular las métricas.

In [1]:
import json
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

dataset_path = Path("data/raw/electoral_dataset.csv")
cwd = Path.cwd()
root = next((path for path in (cwd, *cwd.parents) if (path / dataset_path).is_file()), None)
if root is None:
    raise FileNotFoundError(f"Abre el notebook desde el repositorio que contiene {dataset_path}.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_generation import BALLOT_COLUMNS, GROUND_TRUTH_PATH, RAW_DATA_PATH, generate_dataset
from src.main import run_pipeline
from src.preprocessing import validate_dataset

id_types = {"record_id": "string", "station_id": "string"}
dataset = pd.read_csv(RAW_DATA_PATH, dtype=id_types)
labels = pd.read_csv(GROUND_TRUTH_PATH, dtype={"record_id": "string"})
output_dir = root / "outputs/temp/luna/notebook"

team_dir = output_dir / "team"
run_pipeline(dataset, labels, team_dir)
team_results = team_dir / "results"
report = json.loads((team_results / "metrics.json").read_text(encoding="utf-8"))
prepared = pd.read_csv(team_results / "processed_dataset.csv", dtype=id_types)
display(prepared[["record_id", "station_id", "registered_voters", "total_votes", "participation_rate"]].head())

,record_id,station_id,registered_voters,total_votes,participation_rate
0,REC-0001,ST-0001,299,232,0.775920
1,REC-0002,ST-0002,676,577,0.853550
2,REC-0003,ST-0003,610,432,0.708197
3,REC-0004,ST-0004,491,312,0.635438
4,REC-0005,ST-0005,488,261,0.534836


## Resumen descriptivo

El resumen exportado distingue tasas finitas, faltantes, infinitas e inválidas. Las estadísticas se calculan con tasas finitas, los cuartiles usan interpolación lineal y la desviación estándar es muestral. Los valores indefinidos aparecen como `null`.

In [2]:
summary = report["analysis"]
print(json.dumps(summary, indent=2, allow_nan=False))
rates = summary["participation_rate"]
print(f"En estos datos, la mitad central de las tasas finitas está entre {rates['q1']:.2%} y {rates['q3']:.2%}.")

{
  "record_count": 500,
  "finite_participation_count": 500,
  "missing_participation_count": 0,
  "nonfinite_participation_count": 0,
  "invalid_participation_count": 0,
  "over_100_percent_count": 4,
  "participation_rate": {
    "mean": 0.7233424512666745,
    "median": 0.726189733320475,
    "min": 0.4011142061281337,
    "max": 1.0038167938931297,
    "q1": 0.6309086054317974,
    "q3": 0.8177334915538182,
    "std": 0.1076453133109199
  }
}
En estos datos, la mitad central de las tasas finitas está entre 63.09% y 81.77%.


## Detección estadística y evaluación

El rango intercuartílico es IQR = Q3 − Q1. Con al menos cuatro tasas finitas, el detector alerta únicamente por debajo de Q1 − 1,5 × IQR o por encima de Q3 + 1,5 × IQR. Si IQR es cero, los límites coinciden. Los `record_id` deben ser únicos y los `station_id` pueden repetirse.

El estado `diana_validation_baseline` indica que el pipeline usó la validación de Diana como línea base de reglas. Las métricas de la fila estadística corresponden al IQR.

In [3]:
alerts = pd.read_csv(team_results / "alerts.csv")
statistical = alerts.loc[alerts["method"].eq("statistical")]
iqr = rates["q3"] - rates["q1"]
print(f"Modo: {report['mode']} | Etapas: {report['stages']}")
print(f"Límites IQR: {rates['q1'] - 1.5 * iqr:.2%} a {rates['q3'] + 1.5 * iqr:.2%}")
print(f"Alertas IQR en los datos de Diana: {len(statistical)}")
display(statistical)
display(pd.DataFrame([report["breakdown"]["by_method"]["statistical"]], index=["IQR"]))

Modo: partial | Etapas: {'rule': 'diana_validation_baseline', 'statistical': 'executed', 'analysis': 'executed', 'visualizations': 'pending'}
Límites IQR: 35.07% a 109.80%
Alertas IQR en los datos de Diana: 0


,record_id,method,reason


,TP,FP,FN,TN,precision,recall,f1,confusion_matrix
IQR,0,0,25,475,0.0,0.0,0.0,"[[475, 0], [25, 0]]"


## Caso controlado con participación del 5 %

Se generan 240 registros y cuatro anomalías con el generador de Diana y la semilla 19. El último registro recibe 500 electores y 25 votos, todos para el candidato A. Se agrega una etiqueta `low_turnout` para que Renzo pueda medir la detección. El pipeline escribe este escenario en `low-turnout/`.

In [4]:
controlled, controlled_labels = generate_dataset(n_records=240, n_anomalies=4, seed=19)
row = controlled.index[-1]
record_id = controlled.at[row, "record_id"]
assert record_id not in set(controlled_labels["record_id"])

controlled.loc[row, list(BALLOT_COLUMNS)] = [25, 0, 0, 0, 0]
controlled.loc[row, ["registered_voters", "total_votes"]] = [500, 25]
controlled_labels.loc[len(controlled_labels)] = [record_id, "low_turnout"]
assert record_id not in set(validate_dataset(controlled)["record_id"])

controlled_dir = output_dir / "low-turnout"
controlled_report = run_pipeline(controlled, controlled_labels, controlled_dir)
controlled_alerts = pd.read_csv(controlled_dir / "results/alerts.csv")
target = controlled_alerts.loc[
    controlled_alerts["record_id"].eq(record_id) & controlled_alerts["method"].eq("statistical")
]
assert len(target) == 1

votes = controlled.at[row, "total_votes"]
registered = controlled.at[row, "registered_voters"]
print(f"{record_id} | {votes} votos coherentes / {registered} electores = {votes / registered:.2%}")
print(target.to_string(index=False))
breakdown = controlled_report["breakdown"]
display(pd.DataFrame([item for item in breakdown["by_anomaly_type"] if item["anomaly_type"] == "low_turnout"]))

REC-0240 | 25 votos coherentes / 500 electores = 5.00%
record_id      method                                                                reason
 REC-0240 statistical participation_rate 0.05 is below IQR lower bound 0.35477358873813925.


,anomaly_type,injected,rule,statistical,both,combined
0,low_turnout,1,0,1,0,1


El IQR puede detectar una tasa extrema aunque los conteos sean coherentes. Las anomalías de consistencia pueden tener tasas dentro de los límites. Además, diferencias legítimas entre grupos pueden producir falsas alarmas y cambios en la población pueden desplazar los cuartiles. Cada alerta señala un registro para revisar dentro de estos datos simulados.